# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihaseeb0081/flyrank-ml-internship-haseeb/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My Lane as an ML Task

The problem is a ranking task.

Instead of predicting only yes or no, the system ranks content pages from highest refresh priority to lowest refresh priority.

This allows content teams to focus on the most important pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or Proxy

The ideal target would be a label showing whether a page truly needs a refresh.

Since that label is not available in the dataset, I will create a proxy target.

Proxy Rule:

needs_refresh = 1 if impressions_last_30d < impressions_prev_30d

needs_refresh = 0 otherwise

This proxy identifies pages whose search visibility is declining.

In [31]:
import pandas as pd

In [32]:
!git clone https://github.com/ihaseeb0081/flyrank-ml-internship-haseeb.git

fatal: destination path 'flyrank-ml-internship-haseeb' already exists and is not an empty directory.


In [33]:
!find /content/flyrank-ml-internship-haseeb -name "*.csv"

/content/flyrank-ml-internship-haseeb/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship-haseeb/data/raw/content_refresh_anonymized.csv


In [34]:
df = pd.read_csv(
    "/content/flyrank-ml-internship-haseeb/data/raw/content_refresh_anonymized.csv"
)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [35]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [36]:
df["needs_refresh"] = (
    df["impressions_last_30d"] <
    df["impressions_prev_30d"]
).astype(int)

In [37]:
df[
    [
        "content_id",
        "impressions_prev_30d",
        "impressions_last_30d",
        "needs_refresh"
    ]
].head()

,content_id,impressions_prev_30d,impressions_last_30d,needs_refresh
0,content_304f48230142,987,578,1
1,content_a1fb4e703a9e,5915,2501,1
2,content_9aa793d4d895,6089,2382,1
3,content_331d6c4de07b,4206,3626,1
4,content_d99b7a2d90ca,6452,4211,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric

The primary evaluation metric for this task is F1 Score.

F1 Score balances Precision and Recall, making it useful when identifying pages that truly need a refresh.

Business Success Metric:

If content pages recommended by the model are refreshed, they should show improvements in impressions, clicks, and overall traffic.

In [38]:
refresh_rate = df["needs_refresh"].mean() * 100

print(f"Pages needing refresh: {refresh_rate:.2f}%")

Pages needing refresh: 65.72%


In [39]:
refresh_rate = df["needs_refresh"].mean() * 100

print(f"Pages needing refresh: {refresh_rate:.2f}%")

Pages needing refresh: 65.72%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### The Unit of Analysis

The unit of analysis in this dataset is a content page.

Each row represents one content page and contains information about:

- Search impressions
- Clicks
- Sessions
- Engagement metrics
- Content age
- Content characteristics

The model makes predictions at the content page level to identify which pages may need refreshing.

In [40]:
print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (30000, 45)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,needs_refresh
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [41]:
df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "content_age_days",
        "days_since_last_update"
    ]
].head()

,content_id,impressions_90d,clicks_90d,content_age_days,days_since_last_update
0,content_304f48230142,3803,29,187,20
1,content_a1fb4e703a9e,15320,7,445,25
2,content_9aa793d4d895,12581,11,141,20
3,content_331d6c4de07b,11751,58,463,22
4,content_d99b7a2d90ca,19140,24,263,14


### Observation

The dataset contains one row per content page. Each page has performance, engagement, and freshness-related features that can help determine whether the content should be refreshed.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML Beats a Fixed Rule Here

A fixed rule may only consider one factor, such as declining impressions.

However, content performance depends on many factors including content age, click-through rate, search volume, engagement metrics, sessions, and the number of days since the last update.

Machine learning can analyze all of these factors together and identify patterns that are difficult to capture using simple if-else rules.

Therefore, ML can provide better refresh recommendations than a single fixed rule.

### Why ML Beats a Fixed Rule Here

Not every page with declining impressions needs a refresh.

Some pages are seasonal, while others experience temporary traffic fluctuations.

A fixed rule may incorrectly recommend refreshing such pages.

Machine learning can learn from historical patterns and make more accurate decisions by considering multiple features at the same time.

### Why ML Beats a Fixed Rule Here

A rule-based system is limited because it follows predefined conditions.

Content performance is influenced by many interacting variables, making it difficult to create a perfect set of rules.

Machine learning can discover hidden relationships in the data and rank refresh opportunities more effectively.

In [42]:
important_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update"
]

print("Example features used for decision making:")
print(important_features)

Example features used for decision making:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.